# Paired statistics and OCR/STT availability

Read the archived 100,000-replicate paired inference and calculate descriptive availability-state effects without launching new bootstrap experiments.

**Scope:** These notebooks recompute descriptive statistics from the archived data and predictions. They do not retrain the original encoders or produce new evaluation populations.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd().resolve()
if not (ROOT / "CITATION.cff").exists():
    ROOT = ROOT.parent
assert (ROOT / "CITATION.cff").exists(), "Run from the repository root or notebooks/"
print("Project:", ROOT.name)

## Focal four-contrast source-group inference
The exact 100,000-replicate intervals and permutation values are stored results. This notebook displays them without relabelling the analysis as preregistered.

In [ ]:
stat=pd.read_csv(ROOT / "results/statistics/FOCAL_FOUR_100000.csv")
display(stat[["candidate","baseline","delta_macro_f1","ci_low","ci_high","p_holm_four","replicates","groups"]])
focus=stat[(stat.candidate=="equal_vat")&(stat.baseline=="text")].iloc[0]
assert abs(focus.delta_macro_f1-(-0.0033623179238975))<1e-12
assert focus.replicates==100000

## OCR/STT availability on shared seed-42 observations
Groups can contribute to more than one state. These are descriptive subgroup scores, not independent external populations.

In [ ]:
p=pd.read_csv(ROOT / "results/video/gold_adjudicated_v1.1/VIDEO_OOF_PREDICTIONS.csv")
s=p[p.seed==42].copy()
s["state"]=np.select([(~s.ocr_ok.astype(bool))&(~s.stt_ok.astype(bool)),(~s.ocr_ok.astype(bool))&s.stt_ok.astype(bool),s.ocr_ok.astype(bool)&(~s.stt_ok.astype(bool)),s.ocr_ok.astype(bool)&s.stt_ok.astype(bool)],["Neither","STT only","OCR only","Both"],default="Unknown")
rows=[]
for name,x in s.groupby("state"):
    a=f1_score(x.label_final,x.text_prediction,average="macro",labels=[0,1])
    b=f1_score(x.label_final,x.equal_vat_prediction,average="macro",labels=[0,1])
    rows.append({"availability":name,"n":len(x),"groups":x.group_id.nunique(),"text_f1":a,"equal_vat_f1":b,"delta_vat_minus_text":b-a})
display(pd.DataFrame(rows).sort_values("availability"))
assert len(s)==3951

## Frozen seed variability for main fitted models

In [ ]:
three=pd.read_csv(ROOT / "results/video/gold_adjudicated_v1.1/THREE_SEED_SUMMARY.csv")
display(three[three.model.isin(["text","equal_vat"])][["model","macro_f1_mean","macro_f1_std"]])